In [1]:
!pip install -q transformers accelerate

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [3]:
print("Loading AI model...")
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, 
    device_map="auto"
)
print("Engine loaded.")

Loading AI model...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Engine loaded.


In [6]:
npc_database = {
    "Charles": {"personality": "Extremely impatient, always checking his watch, hates wasting time. Very short and annoyed sentences.", "trust_threshold": 80},
    "George": {"personality": "Very talkative. Loves to ramble and share TMI.", "trust_threshold": 40},
    "Leon": {"personality": "Grumpy and easily annoyed. Complains a lot.", "trust_threshold": 60},
    "Lucia": {"personality": "Mysterious and cryptic. Speaks in riddles and whispers.", "trust_threshold": 60},
    "Rose": {"personality": "Warm, friendly, and caring mother figure.", "trust_threshold": 20}
}

current_npc = "Charles"
current_hour = 14
player_stamina = 25
player_money = 0
trust_level = 75

if 6 <= current_hour < 12:
    time_info = "Morning. Birds are chirping."
elif 12 <= current_hour < 17:
    time_info = "Afternoon. A cool breeze is blowing."
else:
    time_info = "Night. Insects are crying."

npc_data = npc_database[current_npc]

print(f"--- Chatting with {current_npc} ---")
print(f"[STATUS] Time: {current_hour}:00 | Stamina: {player_stamina}/100 | Money: {player_money} | Trust: {trust_level}/{npc_data['trust_threshold']}\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ['quit', 'exit']: break

    system_prompt = f"""
    [ROLE] {current_npc}
    [PERSONALITY] {npc_data['personality']}
    [ENVIRONMENT] Time: {time_info}
    [PLAYER DATA] Stamina: {player_stamina}/100 | Money: {player_money} | Trust: {trust_level}
    
    [LOGIC]
    1. If Stamina <= 30, address the exhaustion first in character.
    2. If Money == 0, mention gathering or selling items.
    3. Reflect the time/atmosphere.
    4. If trust < {npc_data['trust_threshold']}, be cold.
    
    [CONSTRAINTS]
    - Language: English Only.
    - Style: Max 2 sentences. No AI greetings.
    """

    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_input}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=64,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    print(f"{current_npc}: {response}\n")

--- Chatting with Charles ---
[STATUS] Time: 14:00 | Stamina: 25/100 | Money: 0 | Trust: 75/80



You:  Hello I`m Dongmyung


Charles: Hello! How can I assist you today?



You:  look my status


Charles: Your stamina is at 25 out of 100, you have no money, and your trust with me is currently at 75%. How can I assist you today?



...

You:  base on my stamina and money


Charles: I'm sorry to hear you're feeling tired. Let's gather some resources before we head out.



You:  if my stamina is low what do you say?


Charles: I'm sorry to hear that your stamina is low. It's important to rest soon so you can keep going strong!



You:  exit
